# Stochastic resonance: Colab walkthrough

Reproduces the core results in a few minutes: the three noise regimes, the
threshold detector, and a reduced version of the headline SNR sweep.

Everything calls `srlab`. This notebook defines no solver and no SNR estimator
of its own, so the numbers here come from the same code that produced the
figures in `figures/`.

Trial counts and sweep sizes are reduced for runtime. The committed figures use
the full settings in `configs/default.yaml`.

In [ ]:
# Setup. Works in Colab or from a local checkout.
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/<your-user>/sr-project.git"

if "google.colab" in sys.modules:
    if not Path("sr-project").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    root = Path("sr-project")
else:
    root = Path.cwd().parent

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)], check=True)

import numpy as np
import matplotlib.pyplot as plt
import srlab
from srlab import plotting, signals
from srlab.bistable import simulate
from srlab.metrics import bootstrap_snr, snr_db
from srlab.theory import barrier_height, critical_amplitude, two_state_snr_db
from srlab.threshold import threshold_detector

plotting.use_style()
print("srlab", srlab.__version__, "from", root)

## Parameters\n\nRead from the config, so this notebook cannot drift from the experiments.

In [ ]:
import yaml

with open(root / "configs" / "default.yaml") as f:
    cfg = yaml.safe_load(f)

a, b = cfg["system"]["a"], cfg["system"]["b"]
A, f0 = cfg["signal"]["A"], cfg["signal"]["f0"]
dt, fs, decimate = cfg["solver"]["dt"], cfg["solver"]["fs"], cfg["solver"]["decimate"]

A_c = float(critical_amplitude(a, b))
barrier = float(barrier_height(a, b))
print(f"A     = {A}  (A_c = {A_c:.4f}, so A is {A / A_c:.0%} of threshold)")
print(f"dU    = {barrier}")
print(f"peak predicted at D = dU/2 = {barrier / 2}")

## 1. The three regimes

Too little noise and the particle stays put. Too much and it hops at random.
In between it hops in step with the drive.

In [ ]:
D_regimes = cfg["experiments"]["e1"]["D"]
t, x = simulate(a=a, b=b, A=A, f0=f0, D=D_regimes, dt=dt, n_periods=16,
                discard_periods=cfg["solver"]["discard_periods"],
                decimate=decimate, n_trials=8, seed=cfg["seeds"]["e1"])

snr, _, _ = snr_db(x, fs, f0, n_side=5)
show = t <= 6 / f0
clean = signals.sine(1.0, f0, t[show])

fig, axes = plt.subplots(len(D_regimes), 1, figsize=(9, 6), sharex=True)
for i, D in enumerate(D_regimes):
    axes[i].plot(t[show], x[i, 0][show], color=plotting.COLORS["sim"], lw=0.9)
    axes[i].plot(t[show], clean, color=plotting.COLORS["clean"], ls="--", lw=1.5)
    axes[i].set_ylim(-2.2, 2.2)
    axes[i].set_ylabel("x")
    axes[i].set_title(f"D = {D}: SNR = {float(snr[i]):.1f} dB", loc="left")
axes[-1].set_xlabel("time (model units)")
fig.tight_layout()

## 2. A threshold detector shows the same thing

No double well, no dynamics. The signal sits below the threshold, so without
noise the output is empty.

In [ ]:
e2 = cfg["experiments"]["e2"]
sigma = np.logspace(np.log10(e2["sigma_min"]), np.log10(e2["sigma_max"]), 24)
t2 = signals.time_axis(f0, fs, 32)
sig = signals.sine(e2["A"], f0, t2)

snr_thr = []
for s in sigma:
    y = threshold_detector(sig, s, e2["theta"], 10, seed=1)
    # At low sigma the detector never fires, so the record is constant and SNR
    # is undefined. That is the point of the experiment, not an error: report
    # the floor instead. srlab.sweep.chunked_snr does the same thing.
    if y.std() == 0:
        snr_thr.append(-30.0)
    else:
        value, _, _ = snr_db(y, fs, f0, n_side=10)
        snr_thr.append(float(np.ravel(value)[0]))
snr_thr = np.array(snr_thr)

k = int(np.argmax(snr_thr))
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.semilogx(sigma, snr_thr, "o-", ms=3, color=plotting.COLORS["sim"])
ax.axvline(sigma[k], color=plotting.COLORS["optimal"], ls="--",
           label=f"peak at sigma = {sigma[k]:.2f}")
ax.set_xlabel("noise standard deviation sigma")
ax.set_ylabel("output SNR at f0 (dB)")
ax.legend()
fig.tight_layout()
print(f"peak {snr_thr[k]:.1f} dB at sigma = {sigma[k]:.3f}")

## 3. The headline curve, reduced

The committed figure uses 40 noise values and 50 trials over 128 periods. This
runs 16 values and 10 trials over 32 periods, so expect a noisier curve in the
same place. The theory curve is overlaid.

In [ ]:
n_periods, n_trials = 48, 12
D = np.logspace(np.log10(cfg["noise"]["D_min"]), np.log10(cfg["noise"]["D_max"]), 24)

_, x = simulate(a=a, b=b, A=A, f0=f0, D=D, dt=dt, n_periods=n_periods,
                discard_periods=cfg["solver"]["discard_periods"], decimate=decimate,
                n_trials=n_trials, seed=cfg["seeds"]["e3"])
snr, lo, hi = bootstrap_snr(x, fs, f0, n_side=25, n_boot=100)

theory = two_state_snr_db(a, b, A, D, f0, n_periods)
k = int(np.argmax(snr))

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.fill_between(D, lo, hi, color=plotting.COLORS["sim"], alpha=0.2)
ax.semilogx(D, snr, "o-", ms=4, color=plotting.COLORS["sim"], label="simulation")
ax.semilogx(D, theory, color=plotting.COLORS["theory"], label="two-state theory")
ax.axvline(barrier / 2, color=plotting.COLORS["optimal"], ls="--",
           label=f"predicted D = {barrier / 2}")
ax.set_ylim(snr.min() - 5, max(snr.max(), theory.max()) + 3)
ax.set_xlabel("noise intensity D (dimensionless)")
ax.set_ylabel("output SNR at f0 (dB)")
ax.legend(loc="lower center")
fig.tight_layout()

err = 100 * abs(D[k] - barrier / 2) / (barrier / 2)
print(f"measured peak D = {D[k]:.4f} at {snr[k]:.2f} dB")
print(f"predicted       {barrier / 2}")
print(f"error           {err:.1f} %  (grid step here is {100 * (D[1] / D[0] - 1):.0f} %)")

## Notes

The reduced sweep has a coarse grid, so the measured peak lands on whichever
grid point is nearest the true optimum; read the location, not the precision.

Two things this notebook deliberately shows as they are. The curve turns upward
at the lowest noise, which is the intra-well response the two-state model omits
and is discussed in the report. And the theory curve sits a couple of dB above
the simulation, an unresolved prefactor convention, also in the report.

If a result contradicts the theory, that is a finding to report rather than a
parameter to adjust.